In [1]:
%load_ext autoreload
%autoreload 2
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))
import sys
sys.path.insert(0, '/home/kat/Repos/SALSA/')

### 1) Compute property covariance matrix from a sample of ChEMBL.

In [2]:
from rdkit.Chem import Descriptors
import torch
from rdkit import Chem
from rdkit.ML.Descriptors.MoleculeDescriptors import MolecularDescriptorCalculator

desc_names = sorted([x[0] for x in Descriptors._descList])
print(len(desc_names))

def get_props(smiles):
    desc_names = sorted([x[0] for x in Descriptors._descList])
    calc = MolecularDescriptorCalculator(desc_names)
    props = calc.CalcDescriptors(Chem.MolFromSmiles(smiles))
    return torch.nan_to_num(torch.tensor(props),0.0).float()

208


In [3]:
import pandas as pd
import os
import numpy as np
seedy = 666

## Load ChEMBL.
ddir = '/home/kat/Repos/SALSA/data/'
df = pd.read_csv(os.path.join(ddir,'chembl_valid_lte_120.csv'))
# display(df)

## Get sample...
# # # # # #
size = 10000 
# # # # # #
df_samp = df.sample(size, random_state=seedy)
df_samp = df_samp.reset_index(drop=True)
# display(df_samp)

## Calculate MolBert properties on sample.
from tqdm import tqdm
# from tqdm.notebook import tqdm
from property_predictors import surface_predictor

all_props = []
for sm in tqdm(df_samp.smiles.values,total=len(df_samp)):
    props = get_props(sm)
    all_props.append(props)
all_props = np.vstack([np.array(x) for x in all_props])

100%|██████████| 10000/10000 [02:41<00:00, 61.81it/s]


### 2) Get properties for all training set compounds.

In [4]:
## Load in training set.
from contra_seq_dataset import get_dataset_array, get_anc_map

home = '/home/kat/Repos/SALSA/'
anc_path = f'{home}data/model_ready/01/train/anchor_smiles.csv'
aug_path = f'{home}data/model_ready/01/train/augmented_smiles.csv'

df = get_dataset_array(anc_path, aug_path)
anc_map = get_anc_map(df)

## Calculate props for the training set.
from joblib import Parallel, delayed

parallelizer = Parallel(n_jobs=-1, backend= 'multiprocessing' )
tasks = (delayed(get_props)(sm) for sm in df.smiles.values)
train_props = parallelizer(tasks)

In [5]:
train_props = np.vstack([np.array(x) for x in train_props])

### 3) Get rid of all-zero and inf-containing properties in both the training set and chembl.

In [6]:
## Get rid of all-zero properties in chembl set. (to ensure det!=0)
to_drop = []
for i in range(all_props.shape[1]):
    s = sum(all_props[:,i])
    if s==0:
        to_drop.append(i)

## Get rid of properties with weird values (inf, nan, etc.)
for i in range(train_props.shape[1]):
    if np.isinf(train_props[:,i]).any():
        to_drop.append(i) 

In [8]:
to_keep = [i for i in range(all_props.shape[1]) if i not in to_drop]
len(set(to_drop)), len(to_keep)

201

### 4) Get subsets for each prop array, and then compute covariance matrix.

In [10]:
train_props = np.vstack([np.array(x) for x in train_props])
train_props = train_props[:,to_keep]

In [11]:
chembl_props = all_props[:,to_keep]
print(chembl_props.shape)

## Calculate covariance matrix.
cov = np.cov(chembl_props.T)
inv_cov = np.linalg.inv(cov) # inverse of cov matrix 
print(inv_cov.shape)

(10000, 201)
(201, 201)


In [ ]:
chembl_props = all_props[:,to_keep]
print(chembl_props.shape)

## Calculate covariance matrix.
cov = np.cov(chembl_props.T)
inv_cov = np.linalg.inv(cov) # inverse of cov matrix 
print(inv_cov.shape)

In [ ]:
train_props = np.vstack([np.array(x) for x in train_props])
train_props = train_props[:,to_keep]

In [ ]:
## Get rid of properties with weird values (inf, nan, etc.)

keep_list = []
for i in range(train_props.shape[1]):
    if np.isinf(train_props[:,i]).any():
        continue
    else:
        keep_list.append(i)

In [ ]:
chembl_props = all_props[:,to_keep]
train_props = train_props[:,to_keep]

In [ ]:
len(keep_list)

In [ ]:
train_props.shape

In [ ]:
np.isnan(train_props).any()
np.max(train_props)

### 3) Get Mahalanobis distances between each anc-ang pair in the training set w.r.t the previously calculated ChEMBL property covariance matrix.

In [ ]:
from scipy.spatial import distance
from itertools import combinations as combo

# # # # # # # # # # # # # #
far_thresh = 10
# # # # # # # # # # # # # #

anc_aug_dists = []
far_pairs = []
for anc,augs in tqdm(anc_map.items(), total=len(anc_map)):
    augs = list(augs)
    augs.pop(0)
    
    prop_anc = train_props[anc]    
    for aug in augs:
        prop_aug = train_props[aug]
        d = distance.mahalanobis(prop_anc, prop_aug, VI=inv_cov)
        if d > far_thresh:
            far_pairs.append([anc,aug])
        anc_aug_dists.append(d)
        
dist_arr = np.array(anc_aug_dists)  
print(len(dist_arr))
print(len(far_pairs))

In [ ]:
dist_arr

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
sns.set_theme(style='ticks',font_scale=1.25,palette='muted')

plt.figure(figsize=(20,5))
sns.histplot(dist_arr, fill=False, element="step", kde=False) #, bw_adjust=.25)
plt.show()

### 4) Investigate "different" anc-aug pairs.

In [ ]:
far_anc_to_augs = {}
for anc,aug in far_pairs:
    if anc in far_anc_to_augs.keys():
        far_anc_to_augs[anc].append(aug)
    else:
        far_anc_to_augs[anc] = [aug]

In [ ]:
far_ancs = far_anc_to_augs.keys()
df_far_ancs = pd.DataFrame(far_ancs,columns=['idx'])
df_far_ancs['smiles'] = df_far_ancs.apply(lambda x: df.smiles.values[x])
df_far_ancs['len'] = df_far_ancs.smiles.apply(lambda x: len(x))
df_far_ancs = df_far_ancs.sort_values(by='len', ascending=True, ignore_index=True)
df_far_ancs

In [ ]:
from rdkit.Chem import AllChem
from rdkit import Chem, Geometry
from rdkit.Chem import Draw
from rdkit.Chem.Draw import IPythonConsole

def sm_and_mol(sm):
    mol = Chem.MolFromSmiles(sm)
    sm = Chem.MolToSmiles(mol)
    display(sm,mol)
    
    
smis = df.smiles.values

for i,row in df_far_ancs.iterrows():
    
    anc_idx = row.idx
    anc_smi = smis[anc_idx]
    
    mols = [Chem.MolFromSmiles(anc_smi)]

    aug_idc = far_anc_to_augs[anc_idx]
    for aug_idx in aug_idc:
        aug_smi = smis[aug_idx]
        mols.append(Chem.MolFromSmiles(aug_smi))
        
    img = Draw.MolsToGridImage(mols, molsPerRow=len(mols), subImgSize=(250, 250))
    display(img)

In [ ]:
    anc_idx, aug_idx in far_pairs:
#     print(anc_map[i])
#     print(smis[anc_idx])
#     print(smis[aug_idx])

    anc_smi = smis[anc_idx]
    aug_smi = smis[aug_idx]
    mols = [Chem.MolFromSmiles(anc_smi), Chem.MolFromSmiles(aug_smi)]
    img = Draw.MolsToGridImage(mols, molsPerRow=2, subImgSize=(250, 250))
    display(img)

In [ ]:
df